# Chapitre 1.6 : l'évaluation

Un modèle de langage ne « répond » pas à un examen : il donne une probabilité à chaque texte possible. Évaluer « ton IA », c'est donc **lire ces probabilités et compter**. Dans ce carnet, tu refais à la main les mesures de la forge (le moteur de « ton IA ») sur un tout petit modèle, et tu les compares à celles de la version d'une heure, le modèle « mini ».

Ce carnet accompagne [la page du chapitre 1.6](https://chipat-neko.github.io/forge-ia/partie-1/1-6/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. calculer la perte et la perplexité à la main, et les comparer à PyTorch ;
2. les ramener à des bits par octet, pour comparer des modèles qui découpent le texte différemment ;
3. noter un QCM sans que le modèle écrive un mot ;
4. calculer la marge d'erreur et le verdict : la mesure dépasse-t-elle le hasard ?
5. lire des échantillons de texte, et les jetons que le modèle juge les plus probables.

**Comment l'utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet calcule sur le processeur : le modèle est minuscule, et le calcul sur le processeur donne partout les mêmes nombres.

## Étape 0 : récupérer le code, et entraîner un petit modèle

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont le tokenizer a besoin, puis indique à Python où trouver le projet, celui du chapitre 1.5 (qui sait préparer un petit entraînement) et forge.

Elle entraîne ensuite le modèle de l'essai du chapitre 1.5 : 2 couches, des vecteurs de 64 nombres, **60 pas**, quelques secondes. Ce modèle a lu un seul petit texte, pendant très peu de temps : il est très mauvais, et c'est ce qui rend les mesures instructives. Elle charge aussi les relevés de la version d'une heure de « ton IA » (le modèle « mini »), écrits par le projet.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.6-evaluation"
projet_1_5 = depot / "projets" / "1.5-pre-entrainement"
for dossier_du_projet in (projet, projet_1_5):
    if not dossier_du_projet.exists():
        raise FileNotFoundError(
            f"Le dossier {dossier_du_projet} est introuvable. Si le dépôt forge-ia était déjà là, "
            "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
        )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute ce projet, celui du chapitre 1.5, et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str(projet_1_5.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import json
import math
import shutil
import tempfile

import torch
import torch.nn.functional as F
from forge.entrainement import ChargeurJetons
from forge.evaluation.echantillons import echantillons
from forge.evaluation.qcm import QCM, scores_qcm
from forge.modele import nombre_parametres
from forge.tokenizer import TokenizerOctets

from evaluation_pas_a_pas.pas_a_pas import (
    bits_par_octet_a_la_main,
    choix_prefere,
    marge_95,
    marge_hasard,
    octets_du_choix,
    perplexite_a_la_main,
    perte_a_la_main,
    scores_a_la_main,
    scores_par_octet,
    verdict,
)
from evaluation_pas_a_pas.regarder import suivants
from evaluation_pas_a_pas.sauvegarde import charger_sauvegarde
from pre_entrainement.essai import TOKENIZER, essai_pause_reprise, preparer_essai, texte_du_site

appareil = "cpu"  # le modèle est minuscule : le processeur suffit, et donne partout les mêmes nombres


def nombre(x, decimales=0):
    # À la française : une espace fine entre les milliers, une virgule pour les décimales.
    return f"{x:,.{decimales}f}".replace(",", "\u202f").replace(".", ",")


def points(proportion):
    # Une marge en points de pourcentage\u202f: « 0,9 point », « 2,5 points » (le pluriel commence à 2).
    valeur = 100 * proportion
    return f"{nombre(valeur, 1)} point{'s' if valeur >= 2 else ''}"


# Un dossier à nous pour les données, le journal et les sauvegardes de l’essai.
racine = Path(tempfile.mkdtemp(prefix="evaluation-"))
dossier = preparer_essai(racine, texte_du_site())  # les jetons du texte du site, encodés par le tokenizer de « ton IA »
essai = essai_pause_reprise(appareil, racine)  # l’essai du chapitre 1.5\u202f: 60 pas, d’un trait puis avec une pause
modele, config_modele = charger_sauvegarde(racine / "d-un-trait" / "sauvegardes", appareil)
tokenizer = TokenizerOctets.charger(TOKENIZER)
print(f"Ce carnet calcule sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}.")
print(f"Le petit modèle a {nombre(nombre_parametres(config_modele))} paramètres, après {essai['pas_total']} pas d’entraînement.")

# Les relevés de la version d’une heure (« mini »), écrits par `python -m evaluation_pas_a_pas`.
mini = json.loads((projet / "mesures" / "evaluation-mini.json").read_text(encoding="utf-8"))
qcm_mini = json.loads((projet / "mesures" / "qcm-mini.json").read_text(encoding="utf-8"))
print(f"Le modèle « mini » a été évalué au pas {nombre(mini['pas'])}, relevé le {mini['releve_le']}.")

## Étape 1 : la perte et la perplexité, à la main

On donne au modèle des textes qu'il n'a **jamais appris** (la partie `val` des données), et pour chaque position on regarde la probabilité qu'il donne au vrai jeton suivant. La **perte** est la moyenne de `-ln(probabilité)` : si le modèle donne toujours 1 au bon jeton, elle vaut 0 ; plus il en donne peu, plus elle monte. Un modèle qui devine au hasard parmi les 32 768 jetons a une perte de `ln 32 768`, environ 10,4.

La **perplexité** est `exp(perte)` : le modèle hésite comme s'il choisissait au hasard entre autant de jetons également probables.

La cellule d'abord regarde un seul jeton, puis refait toute la perte avec `perte_a_la_main`, qui n'utilise pas `F.cross_entropy`, et compare.

In [ ]:
valeur = ChargeurJetons(dossier / "val", config_modele.longueur_contexte, 16, 0)
x, y = valeur.lot()  # 16 textes de 64 jetons, et pour chaque position le vrai jeton suivant
with torch.no_grad():
    logits, _ = modele(x)  # un score par jeton possible, à chaque position
logits = logits.double()  # en 64 bits, pour comparer à PyTorch au chiffre près

# Un seul jeton\u202f: la position 10 du premier texte.
probas = torch.softmax(logits[0, 10], dim=-1)
vrai = y[0, 10].item()
print(f"Le vrai jeton suivant est {tokenizer.octets_du_jeton(vrai).decode('utf-8', errors='replace')!r}.")
print(f"Le modèle lui donne la probabilité {nombre(probas[vrai].item(), 6)}, soit -ln = {nombre(-math.log(probas[vrai].item()), 3)}.")

# Tous les jetons\u202f: la moyenne de ces -ln.
perte = perte_a_la_main(logits, y)
reference = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1)).item()
print(f"Perte à la main\u202f: {nombre(perte, 6)}")
print(f"Perte de PyTorch\u202f: {nombre(reference, 6)}")
print(f"Elles sont égales à 10⁻¹² près\u202f: {abs(perte - reference) <= 1e-12}")

hasard = math.log(config_modele.taille_vocabulaire)
perplexite = perplexite_a_la_main(perte)
print(f"Un modèle qui devine au hasard\u202f: ln {nombre(config_modele.taille_vocabulaire)} = {nombre(hasard, 3)}")
print(f"Perplexité\u202f: {nombre(perplexite, 1)}, comme un choix au hasard entre {nombre(perplexite)} jetons sur {nombre(config_modele.taille_vocabulaire)}.")
print(f"Pour « mini », la perte de validation est {nombre(mini['perte_val'], 3)} et la perplexité {nombre(mini['perplexite'], 1)}.")

## Étape 2 : les bits par octet

La perte se mesure en nats **par jeton**. Mais un jeton n'a pas la même longueur d'un tokenizer à l'autre : un tokenizer qui découpe en gros morceaux obtient une perte plus haute par jeton, sans être pire. Pour comparer, on ramène la perte à **un octet de texte** : on la divise par `ln 2` (pour passer des nats aux bits), puis par le nombre moyen d'octets d'un jeton.

La cellule compte les octets des jetons qu'on vient de lire, puis calcule les bits par octet du petit modèle, du hasard et de « mini ».

In [ ]:
octets = sum(len(tokenizer.octets_du_jeton(int(i))) for i in y.flatten())
octets_par_jeton = octets / y.numel()
bits = bits_par_octet_a_la_main(perte, octets_par_jeton)
bits_du_hasard = bits_par_octet_a_la_main(hasard, octets_par_jeton)
print(f"Ces {nombre(y.numel())} jetons font {nombre(octets)} octets\u202f: {nombre(octets_par_jeton, 2)} octets par jeton.")
print(f"Bits par octet du petit modèle\u202f: {nombre(bits, 3)}")
print(f"Bits par octet d’un modèle qui devine au hasard\u202f: {nombre(bits_du_hasard, 3)}")
print(f"Bits par octet de « mini »\u202f: {nombre(mini['bits_par_octet'], 3)}")

## Étape 3 : un QCM noté sans que le modèle écrive un mot

Dans un QCM, le modèle ne choisit pas une lettre : on lui fait **lire** la question suivie de chaque réponse possible, et on regarde laquelle lui paraît la plus vraisemblable. Pour une réponse, on additionne les `ln(probabilité)` de ses jetons, sachant tout ce qui précède : c'est sa **log-vraisemblance**.

Une réponse longue a plus de jetons à deviner, donc une log-vraisemblance plus basse : elle perdrait toujours. Pour être juste, on divise aussi chaque score par le nombre d'octets de la réponse, et on compare des nats **par octet**. La part des questions où ce score par octet désigne la bonne réponse, c'est l'exactitude *normalisée* du rapport de forge.

La cellule note une question, avec `scores_a_la_main`, et vérifie que forge (`scores_qcm`) trouve les mêmes nombres. Le modèle n'écrit rien : il lit quatre textes.

In [ ]:
question = QCM(
    "Question : Quelle est la capitale de la France ?\nRéponse :",
    [" Paris.", " Lyon, une grande ville au bord du Rhône.", " Rome.", " Berlin."],
    0,  # la bonne réponse est la première
)
scores = scores_a_la_main(modele, tokenizer, question)
par_octet = scores_par_octet(question, scores)
print(f"{'réponse':<44}{'score':>9}{'octets':>8}{'par octet':>11}")
for choix, score, normalise in zip(question.choix, scores, par_octet):
    print(f"{choix.strip():<44}{nombre(score, 2):>9}{octets_du_choix(choix):>8}{nombre(normalise, 3):>11}")
brut, normalise = choix_prefere(scores), choix_prefere(par_octet)
print(f"Réponse préférée sans normaliser\u202f: {question.choix[brut].strip()}")
print(f"Réponse préférée après normalisation\u202f: {question.choix[normalise].strip()}")
print(f"La bonne réponse était\u202f: {question.choix[question.bonne].strip()}")
de_forge = scores_qcm(modele, tokenizer, [question])[0]
print(f"Mêmes scores que forge, à 10⁻⁵ près en relatif\u202f: {all(math.isclose(a, b, rel_tol=1e-5) for a, b in zip(scores, de_forge))}")

# Six questions du jeu MMMLU, notées par « mini » (relevé par `python -m evaluation_pas_a_pas qcm-exemples`).
jeu = qcm_mini["jeu"]
print()
print(f"{qcm_mini['n']} questions de {jeu['nom']} ({jeu['configuration']}), licence {jeu['licence']}\u202f: {jeu['adresse']}")
print(f"« mini » en a trouvé {qcm_mini['justes']} sur {qcm_mini['n']}\u202f; le hasard en donnerait {nombre(qcm_mini['hasard'] * qcm_mini['n'], 1)}.")

## Étape 4 : la marge et le verdict

Sur 5 questions à 4 choix, un tirage au sort en réussit 1 ou 2 : une mesure ne vaut rien sans sa **marge d'erreur**. Deux marges comptent :

- `marge_95(exactitude, n)`, la marge de la *mesure* : l'intervalle à 95 % autour de ce qu'on a mesuré. Elle s'annule quand on a tout juste ou tout faux, même sur 5 questions ;
- `marge_hasard(n, choix)`, la marge du *hasard* : jusqu'où un tirage au sort s'écarte de 1 sur 4, sur ces `n` questions. Elle ne dépend pas du résultat.

Le **verdict** compare l'exactitude normalisée au hasard avec la marge du *hasard* : « au-dessus » si l'écart la dépasse, « en dessous » s'il est plus bas qu'elle, sinon « pas-different ». C'est la règle du rapport de forge. Prendre la marge de la mesure serait un piège : 0 bonne réponse sur 5 ressemblerait à « en dessous du hasard », alors que ça arrive une fois sur quatre au hasard.

In [ ]:
print("Les deux tests français de « mini »\u202f:")
for nom, m in mini["qcm"].items():
    exactitude, n = m["exactitude_normalisee"], m["n"]
    print(f"  {nom}\u202f: {nombre(100 * exactitude, 1)}\u202f% sur {nombre(n)} questions, hasard {nombre(100 * m['hasard'], 1)}\u202f%")
    print(f"    marge de la mesure ± {points(marge_95(exactitude, n))}, marge du hasard ± {points(m['marge_hasard'])}")
    ecart = verdict(exactitude, m["hasard"], m["marge_hasard"])
    print(f"    verdict\u202f: {ecart} (celui du relevé\u202f: {m['verdict']})")

print()
print("Le piège de la petite série\u202f: 0 bonne réponse sur 5 questions à 4 choix.")
print(f"  marge de la mesure ± {points(marge_95(0.0, 5))} (elle s’annule)")
print(f"  marge du hasard ± {points(marge_hasard(5, 4))}")
print(f"  verdict\u202f: {verdict(0.0, 0.25, marge_hasard(5, 4))}")

# À toi\u202f: change ces deux nombres, et relance la cellule.
n, justes = 100, 31
exactitude = justes / n
print()
print(f"{justes} bonnes réponses sur {n} (hasard 25\u202f%)\u202f: {verdict(exactitude, 0.25, marge_hasard(n, 4))}")

## Étape 5 : lire des échantillons

Les chiffres disent si le modèle prédit bien ; les **échantillons** montrent comment il écrit. Forge fait écrire à chaque modèle la suite de dix débuts fixes, avec une graine fixe : d'une sauvegarde à l'autre, on lit le même texte amorcé de la même façon, et on voit ce qui change.

La cellule lit d'abord trois échantillons de « mini » dans son relevé. Elle fait ensuite écrire le petit modèle de ce carnet, puis regarde les jetons qu'il juge les plus probables après une phrase. Lis-les sans te fier à ce qu'ils racontent : un échantillon est un coup d'œil, pas une mesure.

In [ ]:
print("« mini » (la version d’une heure)\u202f:")
for e in mini["echantillons"][:3]:
    print(f"  {e['invite']!r} → {e['suite'][:140].replace(chr(10), ' ')}")

print()
print("Le petit modèle de ce carnet (60 pas)\u202f:")
for e in echantillons(modele, tokenizer, 0, 30)[:2]:
    print(f"  {e['invite']!r} → {e['suite'][:140].replace(chr(10), ' ')!r}")

print()
print("Les 5 jetons les plus probables après « Il était une » pour le petit modèle\u202f:")
for jeton, proba in suivants(modele, tokenizer, "Il était une", k=5):
    print(f"  {jeton!r:<12} {nombre(100 * proba, 2)}\u202f%")

In [ ]:
shutil.rmtree(racine, ignore_errors=True)  # le dossier temporaire de l’essai
print("Ce carnet a tourné sur le processeur.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.6-evaluation sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.6-evaluation).
- La page du chapitre : [chapitre 1.6](https://chipat-neko.github.io/forge-ia/partie-1/1-6/).
- Depuis le dossier du projet, `uv run --extra cpu python -m evaluation_pas_a_pas resumer --rapport <rapport.json>` résume le rapport d'une évaluation de forge, et `qcm-exemples`, `cartes` et `suivants` regardent un modèle sauvegardé.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0. Les questions de MMMLU gardent la licence MIT de leur jeu.